# ExPLoRA Stage 1 + probes on Colab / Kaggle

Runs the **same scripts** as your laptop, with checkpoints on Google Drive (Colab) or `/kaggle/working` (Kaggle), every epoch.
Before running: push your code to GitHub (the notebook clones it), and set **Runtime → GPU (T4)**.

We install with the platform's preinstalled CUDA torch, plus `pip` for the rest. The `uv.lock` pins a torch built for CUDA 13.2, which may not match the Colab/Kaggle driver. Locally, uv stays the package manager.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = "/content/drive/MyDrive/explora/checkpoints/stage1"
else:
    CKPT_DIR = "/kaggle/working/checkpoints/stage1" if IN_KAGGLE else "checkpoints/stage1"
print("checkpoints ->", CKPT_DIR)

In [ ]:
REPO = "https://github.com/Panshul-Jindal/PlayingWithParameterEfficientFineTuning.git"
!git clone -q $REPO repo 2>/dev/null || (cd repo && git pull -q)
%cd repo
!pip install -q "transformers>=5" medmnist wandb
!pip install -q -e . --no-deps
!nvidia-smi --query-gpu=name,memory.total --format=csv

### wandb login
Colab: add a secret `WANDB_API_KEY` (key icon in the sidebar). Kaggle: Add-ons → Secrets → `WANDB_API_KEY`.

In [ ]:
if IN_COLAB:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
elif IN_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

### 0. Sanity: all tests pass on this machine

In [ ]:
!python -m pytest -q

### 1. Baseline probe (Task 1)

In [ ]:
!python scripts/run_probe.py --tag baseline --num-workers 2

### 2. Time one epoch, then train (Task 2). Re-run the training cell after a disconnect: `--resume` continues from the last epoch

In [ ]:
!python scripts/run_stage1.py --time-only --num-workers 2 --out-dir /tmp/timing

In [ ]:
EPOCHS = 20
!python scripts/run_stage1.py --epochs $EPOCHS --num-workers 2 --out-dir $CKPT_DIR --resume

In [ ]:
!mkdir -p /content/drive/MyDrive/explora/run_outputs && cp -r outputs results/runs.csv /content/drive/MyDrive/explora/run_outputs/

### 3. Merge LoRA + probe again (Task 3)

In [ ]:
!python scripts/run_probe.py --tag explora --checkpoint $CKPT_DIR/final.pt --num-workers 2